# Добор недостающей ветки абляции

В прогоне на Colab не досчиталась одна ветка: `--tail_estimate direct` на полной
конфигурации (1000 узлов, 32 члена). Кончилась квота GPU.

Без неё сравнение в разделе 5.9 отчёта отличается двумя параметрами, а не одним:
у ветки с Дейкстрой нет запаса доверия, а у `gpu_full`, с которым её сравнивали,
он включён. Этот ноутбук считает **только** недостающее.

**Ничего из уже посчитанного он не трогает.** Результат надо сравнить с
`gpu_tail_dijkstra = 0.430`, поэтому сиды (1,2,3) и число эпизодов (20) обязаны
совпадать — они зашиты в команду ниже.

Про время. Тот же прогон с `dijkstra` занял по логам **102 минуты**. Здесь
добавлен `--tgt_chunk 2048`: при наборах по 32 члена блок по умолчанию вмещает
`128/32 = 4` узла, то есть один запрос планировщика дробится на 250 запусков
ядра. На замере укрупнение блока дало ускорение втрое, решения при этом не
меняются (расхождение 5e-07, `argmin` совпадает в 100%).

Отсюда ожидание **40–60 минут**, но это экстраполяция замера с CPU. Ячейка
печатает прогноз после первого сида — по нему станет ясно за треть времени.

## 1. Установка

In [1]:
import os

# Абсолютный путь и проверка на существование — иначе повторный запуск ячейки
# клонирует репозиторий ВНУТРЬ уже скачанного и уходит на уровень глубже. Так
# уже случалось: рабочей оказывалась вложенность из трёх одинаковых папок.
REPO = '/content/fb-multi-intention-planning'
URL = 'https://github.com/2FIVE192/fb-multi-intention-planning.git'

if os.path.isdir(os.path.join(REPO, '.git')):
    print('репозиторий уже склонирован, обновляю')
    !cd {REPO} && git pull --ff-only && git submodule update --init --recursive
else:
    !git clone --recursive {URL} {REPO}

%cd {REPO}
!pip install -q -r requirements-colab.txt
print('\nрабочая директория:', os.getcwd())

Cloning into '/content/fb-multi-intention-planning'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (250/250), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 250 (delta 133), reused 207 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (250/250), 223.27 KiB | 7.97 MiB/s, done.
Resolving deltas: 100% (133/133), done.
Submodule 'third_party/switching-successor-measures' (https://github.com/stestoKTH/switching-successor-measures.git) registered for path 'third_party/switching-successor-measures'
Cloning into '/content/fb-multi-intention-planning/third_party/switching-successor-measures'...
remote: Enumerating objects: 48, done.        
remote: Counting objects: 100% (19/19), done.        
remote: Compressing objects: 100% (9/9), done.        
remote: Total 48 (delta 13), reused 10 (delta 10), pack-reused 29 (from 1)        
Receiving objects: 100% (48/48), 133.94 KiB | 4.32 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Submod

## 2. Окружение

In [2]:
import os
import subprocess
import sys

# JAX по умолчанию занимает ~75% видеопамяти при первом же использовании. Если
# это сделает ядро ноутбука, дочерним процессам памяти уже не достанется.
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')


def run(*args, env=None, quiet=False):
    """Запускает скрипт репозитория отдельным процессом.

    Вывод читается построчно и печатается заново через print. Это не
    придирчивость: `subprocess.run` наследует файловые дескрипторы ядра, а в
    Colab вывод ячейки — объект Python поверх ZMQ, а не настоящий дескриптор.
    Без перепечатки весь вывод дочернего процесса уходит в лог сервера.

    `-X faulthandler` заставляет Python напечатать стек при падении в нативном
    коде: без него SIGSEGV не оставляет вообще никаких следов. Именно так и был
    найден источник падения на Colab.

    Функция намеренно ничего не возвращает: значение последнего выражения
    ячейки Jupyter печатает сам, и захваченный вывод дублировался бы сырой
    строкой с видимыми \\n.
    """
    command = [sys.executable, '-X', 'faulthandler', '-u', *args]
    if not quiet:
        print('$', ' '.join(command[4:]), flush=True)

    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1, env=env,
    )
    for line in process.stdout:
        if not quiet:
            print(line, end='', flush=True)
    code = process.wait()

    if code == 0:
        return

    hints = {-9: 'SIGKILL, почти всегда нехватка оперативной памяти',
             -6: 'SIGABRT, падение в нативной библиотеке',
             -11: 'SIGSEGV, падение в нативной библиотеке'}
    raise RuntimeError(f'процесс завершился с кодом {code}'
                       + (f' ({hints[code]})' if code in hints else ''))


# --- Проверка окружения ----------------------------------------------------
# Про графику. OGBench безусловно создаёт mujoco.Renderer в MazeEnv.__init__ и
# сразу рендерит кадр, то есть требует рабочего OpenGL даже когда картинки не
# нужны. На Colab это давало SIGSEGV прямо в mujoco.MjrContext — и на egl, и на
# osmesa, причём только когда до создания среды успевал загрузиться jaxlib с
# CUDA. Подбор backend'а такую поломку не лечит: конфликтуют нативные библиотеки.
#
# Поэтому репозиторий подменяет рендерер заглушкой (fbplan/_upstream.py):
# кадры мы нигде не используем, а источник падения исчезает целиком. Если
# картинки всё же понадобятся, верните настоящий рендерер через
# FBPLAN_KEEP_RENDERER=1 — тогда backend снова придётся подбирать.

try:
    run('scripts/diagnose_env.py', '--probe', 'upstream_path')
except RuntimeError as exc:
    raise RuntimeError(
        f"""Среда не создаётся ({exc}).

Полная диагностика покажет, на каком шаге ломается:
    run('scripts/diagnose_env.py')

Она прогоняет пять проб, отличающихся одним шагом, каждую отдельным процессом
с faulthandler — первая упавшая и называет причину."""
    ) from exc

# GPU проверяем тоже в подпроцессе: ядро не должно трогать видеопамять вообще.
run('-c', 'import jax; print("устройства jax:", jax.devices()); '
          'assert jax.devices()[0].platform == "gpu", '
          '"GPU не подключён: Среда выполнения -> Сменить среду выполнения"')

print('\nокружение готово')

$ scripts/diagnose_env.py --probe upstream_path
    make_env_and_datasets(env_only=True)
    успех
$ -c import jax; print("устройства jax:", jax.devices()); assert jax.devices()[0].platform == "gpu", "GPU не подключён: Среда выполнения -> Сменить среду выполнения"
устройства jax: [CudaDevice(id=0)]

окружение готово


## 3. Данные и чекпоинты

In [3]:
!python scripts/download_datasets.py --datasets antmaze-medium-navigate-v0

[get] https://rail.eecs.berkeley.edu/datasets/ogbench/antmaze-medium-navigate-v0.npz
antmaze-medium-navigate-v0.npz: 100% 232M/232M [00:02<00:00, 94.8MB/s]
[ok] /root/.ogbench/data/antmaze-medium-navigate-v0.npz
[get] https://rail.eecs.berkeley.edu/datasets/ogbench/antmaze-medium-navigate-v0-val.npz
antmaze-medium-navigate-v0-val.npz: 100% 23.2M/23.2M [00:00<00:00, 68.6MB/s]
[ok] /root/.ogbench/data/antmaze-medium-navigate-v0-val.npz

готово: /root/.ogbench/data


In [4]:
!pip -q install gdown
!python -m gdown --folder https://drive.google.com/drive/folders/1dKYhaDJH9lUREo-kUV3AwmTLrxvKO7Ek -O checkpoints

CHECKPOINT = 'checkpoints/medium'
ENV = 'ogbench-antmaze-medium-navigate-v0'

import os
assert os.path.isfile(os.path.join(CHECKPOINT, 'params.pkl')), 'чекпоинт не скачался'
print(sorted(os.listdir(CHECKPOINT)))

Retrieving folder contents
Retrieving folder 1SwRXBwFpsPUNCzC1SoMVKtx540URtJrW giant
Processing file 1ZzEc7ujezMj2U2aUDeeVFp1qUOi1t0Zx flags.json
Processing file 1HOOIW3uQkcwEYEZwLTsXWu-tU7lshkt0 params.pkl
Retrieving folder 1prWCPrmR_TzT9AahjbBrUaLidlycqIWP large
Processing file 19nh-KCENZDYrOa_ME5du4PqNi3BQVlru flags.json
Processing file 1gV70CRaJudM_lkqi3FFYwLTg82DAyodi params.pkl
Retrieving folder 1mg806bd3v28KTm_GUkWQJyPZKvSCOou- medium
Processing file 1r0wR4il8LIbrPKRMakEckCR4MOs7romJ flags.json
Processing file 1683zzDk5v4m2Otad6I2E0TsoYBcGypqB params.pkl
Retrieving folder 1mJZHteZ7mr1WSWuwdoHv5sw-rgO1lcDh teleport
Processing file 1OrHUH4kPwUdQI067sFsq6twMe1oEAE28 flags.json
Processing file 1FCNHFuMlrLJYlNvqJaoaHP2nw0vreWOR params.pkl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1ZzEc7ujezMj2U2aUDeeVFp1qUOi1t0Zx
To: /content/fb-multi-intention-planning/checkpoints/giant

## 4. Недостающий прогон

Первым делом строится граф полной конфигурации (~200 с), он же кэшируется.
Дальше три сида по 100 эпизодов. Следите за строками `[eval] сид N ... с
суммарно`: если первый сид уложится примерно в 700 с, весь прогон займёт около
40 минут; если ближе к 2000 с — значит ускорение не сработало, и будет около
100 минут, как в прошлый раз.

In [5]:
import time

started = time.time()
run('scripts/run_eval.py',
    '--checkpoint_dir', CHECKPOINT, '--env_name', ENV,
    '--methods', 'graph', '--seeds', '1,2,3', '--num_episodes', '20',
    '--replan_every', '20', '--execution', 'high', '--min_commit_steps', '40',
    '--tail_estimate', 'direct',
    '--num_nodes', '1000', '--num_members', '32', '--member_stride', '2',
    '--normalizer_references', '4000',
    '--tgt_chunk', '2048',          # только скорость, на решения не влияет
    '--no_progress', '--tag', 'gpu_tail_direct')

print()
print(f'заняло {(time.time() - started) / 60:.0f} минут')

$ scripts/run_eval.py --checkpoint_dir checkpoints/medium --env_name ogbench-antmaze-medium-navigate-v0 --methods graph --seeds 1,2,3 --num_episodes 20 --replan_every 20 --execution high --min_commit_steps 40 --tail_estimate direct --num_nodes 1000 --num_members 32 --member_stride 2 --normalizer_references 4000 --tgt_chunk 2048 --no_progress --tag gpu_tail_direct
[exp] среда ogbench-antmaze-medium-navigate-v0
[exp] чекпоинт checkpoints/medium
[checkpoint] загружено: checkpoints/medium/params.pkl (epoch=None)
[exp] задача 1: целевых состояний в датасете 1897, взято 64
[exp] задача 2: целевых состояний в датасете 1114, взято 64
[exp] задача 3: целевых состояний в датасете 791, взято 64
[exp] задача 4: целевых состояний в датасете 485, взято 64
[exp] задача 5: целевых состояний в датасете 737, взято 64
E0817 08:00:38.192943    1432 buffer_comparator.cc:150] Difference at 21250946: -0.507812, expected -0.755885
E0817 08:00:38.198215    1432 buffer_comparator.cc:150] Difference at 22068645:

## 5. Сравнение с уже полученной веткой

In [6]:
import shutil

import pandas as pd

df = pd.read_csv('results/raw/gpu_tail_direct_episodes.csv')
direct = df.success.mean()

# Число из прошлого прогона: сырые csv остались в том рантайме и пропали.
DIJKSTRA = 0.430

print('абляция глубины плана, полная конфигурация (1000 узлов, 32 члена)')
print(f'  хвост по Дейкстре   : {DIJKSTRA:.3f}   (из прошлого прогона)')
print(f'  хвост одной подцелью: {direct:.3f}   ({len(df)} эпизодов)')
print()
print(f'разница: {direct - DIJKSTRA:+.3f}')
print()
print('На CPU то же сравнение дало 0.47 против 0.69. Если знак и порядок')
print('величины совпали, вывод раздела 5.6 подтверждается и на лучших рёбрах.')

# Копия в /content: файлы внутри репозитория легко потерять при перезапуске.
shutil.copy('results/raw/gpu_tail_direct_episodes.csv', '/content/gpu_tail_direct_episodes.csv')
print()
print('Копия положена в /content/gpu_tail_direct_episodes.csv — скачайте её через')
print('панель файлов слева. Иначе результат пропадёт вместе с рантаймом, как в')
print('прошлый раз, и числа опять придётся вынимать из вывода ячеек.')

абляция глубины плана, полная конфигурация (1000 узлов, 32 члена)
  хвост по Дейкстре   : 0.430   (из прошлого прогона)
  хвост одной подцелью: 0.630   (300 эпизодов)

разница: +0.200

На CPU то же сравнение дало 0.47 против 0.69. Если знак и порядок
величины совпали, вывод раздела 5.6 подтверждается и на лучших рёбрах.

Копия положена в /content/gpu_tail_direct_episodes.csv — скачайте её через
панель файлов слева. Иначе результат пропадёт вместе с рантаймом, как в
прошлый раз, и числа опять придётся вынимать из вывода ячеек.
